In [0]:
# ==========================================
# Read Bronze Tables
# ==========================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

customer_df = spark.table("bronze_customer")

payment_df = spark.table("bronze_payment")

customer_df.printSchema()
customer_df.show(5, truncate=False)

payment_df.printSchema()
payment_df.show(5, truncate=False)
# ==========================================
# Customer Data Type Casting
# ==========================================

customer_df = (
    customer_df
    .withColumn("CustomerID", col("CustomerID").cast("int"))
    .withColumn("OutstandingBalance", col("OutstandingBalance").cast("double"))
    .withColumn("CreatedDate", col("CreatedDate").cast("date"))
    .withColumn("LastUpdatedDate", col("LastUpdatedDate").cast("date"))
)
customer_df.printSchema()
customer_df.show(5)

# ==========================================
# Payment Data Type Casting
# ==========================================
payment_df = (
    payment_df
    .withColumn("CustomerID", col("CustomerID").cast("int"))
    .withColumn("PaymentDate", col("PaymentDate").cast("date"))
    .withColumn("AmountPaid", col("AmountPaid").cast("double"))
)



customer_df = customer_df.withColumn(
    "CustomerName",
    upper(col("CustomerName"))
)

customer_df.select("CustomerName").show(10, truncate=False)
customer_df = customer_df.withColumn(
    "AccountStatus",
    upper(col("AccountStatus"))
)

customer_df.groupBy("AccountStatus").count().show()

customer_df = customer_df.withColumn(
    "PhoneNumbers",
    regexp_replace(col("PhoneNumbers"), "[()\\-\\s]", "")
)

customer_df.select("PhoneNumbers").show(10)

customer_df.printSchema()

customer_df.show(10)

split(col("PhoneNumbers"), ",").getItem(0)

# ==========================================
# Extract Primary Contact Number
# ==========================================

customer_df = customer_df.withColumn(
    "ContactNumber",
    split(col("PhoneNumbers"), ",").getItem(0)
)

customer_df.select(
    "PhoneNumbers",
    "ContactNumber"
)
customer_df.show(10)

# ==========================================
# Extract STATE
# ==========================================

customer_df = customer_df.withColumn(
    "State",
    split(col("Address"), ",").getItem(2)
)

customer_df = customer_df.withColumn(
    "State",
    trim(col("State")))

customer_df.show()

# ==========================================
# Data Quality Validation
# ==========================================

customer_df = customer_df.withColumn(

    "RejectReason",

    when(col("CustomerID").isNull(), "CustomerID is NULL")

    .when(trim(col("CustomerName")) == "", "Customer Name is Empty")

    .when(
        ~col("ContactNumber").rlike("^[0-9]{10}$"),
        "Invalid Contact Number"
    )

    .when(
        col("OutstandingBalance") < 0,
        "Negative Outstanding Balance"
    )

    .otherwise("VALID")
)

valid_customer_df = customer_df.filter(
    col("RejectReason") == "VALID"
)

reject_customer_df = customer_df.filter(
    col("RejectReason") != "VALID"
)

print("Valid Customers :", valid_customer_df.count())

print("Rejected Customers :", reject_customer_df.count())


# ==========================================
# Payment Aggregation
# ==========================================

payment_summary_df = (
    payment_df
    .groupBy("CustomerID")
    .agg(
        sum("AmountPaid").alias("TotalPayment")
    )
)

payment_summary_df.show(10, truncate=False)

# ==========================================
# Join Customer and Payment Data
# ==========================================

silver_df = (
    valid_customer_df
    .join(
        payment_summary_df,
        "CustomerID",
        "left"
    )
)

silver_df = silver_df.fillna(
    {
        "TotalPayment": 0
    }
)

silver_df.show(10, truncate=False)


# ==========================================
# Calculate Outstanding Balance
# ==========================================

silver_df = silver_df.withColumn(
    "NewOutstandingBalance",
    greatest(
        col("OutstandingBalance") - col("TotalPayment"),
        lit(0)
    )
)

silver_df.select(
    "CustomerID",
    "OutstandingBalance",
    "TotalPayment",
    "NewOutstandingBalance"
).show(10, truncate=False)

# ==========================================
# Update Account Status
# ==========================================

silver_df = silver_df.withColumn(

    "UpdatedAccountStatus",

    when(
        (col("AccountStatus") == "SUSPENDED") &
        (col("NewOutstandingBalance") <= 0),
        "ACTIVE"
    )

    .when(
        (col("AccountStatus") == "PENDING") &
        (col("NewOutstandingBalance") <= 0),
        "ACTIVE"
    )

    .when(
        col("AccountStatus") == "CANCELLED",
        "CANCELLED"
    )

    .otherwise(col("AccountStatus"))
)

silver_df.select(
    "CustomerID",
    "AccountStatus",
    "NewOutstandingBalance",
    "UpdatedAccountStatus"
).show(10, truncate=False)


# ==========================================
# Collections Priority
# ==========================================

silver_df = silver_df.withColumn(

    "CollectionsPriority",

    when(col("NewOutstandingBalance") >= 1000, "HIGH")

    .when(
        (col("NewOutstandingBalance") >= 500) &
        (col("NewOutstandingBalance") < 1000),
        "MEDIUM"
    )

    .otherwise("LOW")
)

silver_df.groupBy("CollectionsPriority").count().show()


# ==========================================
# Finalize Silver Dataset
# ==========================================

silver_df = (
    silver_df
    .drop("OutstandingBalance", "AccountStatus")
    .withColumnRenamed("NewOutstandingBalance", "OutstandingBalance")
    .withColumnRenamed("UpdatedAccountStatus", "AccountStatus")
)

silver_df.printSchema()
silver_df.show(10, truncate=False)

# ==========================================
# Save Silver Table
# ==========================================

silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customer")

reject_customer_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_rejected_customer")






root
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- PhoneNumbers: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- AccountStatus: string (nullable = true)
 |-- OutstandingBalance: string (nullable = true)
 |-- CreatedDate: string (nullable = true)
 |-- LastUpdatedDate: string (nullable = true)

+----------+----------------+--------------------------------+---------------------------------+-------------+------------------+-----------+---------------+
|CustomerID|CustomerName    |PhoneNumbers                    |Address                          |AccountStatus|OutstandingBalance|CreatedDate|LastUpdatedDate|
+----------+----------------+--------------------------------+---------------------------------+-------------+------------------+-----------+---------------+
|100001    |David Williams  |6136505587                      |96 Main St, Los Angeles, CA, USA |Pending      |1204.04           |2025-01-03 |2025-06-18     |
|100002 